# Agentic AI Customer Support Desk

**Meridian Fleet Solutions** — An interactive exercise exploring how agentic AI differs from chatbots and RAG-only systems.

Run each cell in order. The only variable you need to change is `ACTIVE_TICKET` in Section 9.

## 1. Setup

<!-- FACILITATOR: Setup — no pause needed. Walk the room to make sure every participant's cells execute without errors before moving on. -->

Install dependencies, load embeddings, and verify the environment.

In [1]:
from src.cells.setup import initialize, DEMO_MODE

initialize(demo_mode=True)

[SETUP] DEMO_MODE=True — using pre-scripted responses (no API key needed)

╔══════════════════════════════════════════════════════════╗
║  ✅ SETUP COMPLETE                                        ║
╠══════════════════════════════════════════════════════════╣
║  Mode:         DEMO (pre-scripted responses)             ║
║  Embeddings:   Loaded (14 chunks from 5 KB documents)    ║
║  Tools:        7 available                               ║
║    • search_kb                                           ║
║    • query_customer_db                                   ║
║    • send_email                                          ║
║    • escalate                                            ║
║    • request_info                                        ║
║    • check_sla_status                                    ║
║    • check_open_cases                                    ║
╠══════════════════════════════════════════════════════════╣
║  Ready! Change ACTIVE_TICKET and run the cells below.    ║
╚═════════

{'demo_mode': True,
 'api_key': None,
 'embeddings_loaded': True,
 'tools_available': ['search_kb',
  'query_customer_db',
  'send_email',
  'escalate',
  'request_info',
  'check_sla_status',
  'check_open_cases']}

## 2. Knowledge Base

<!-- FACILITATOR: Knowledge Base — Pause briefly. Ask: 'Take a look at these five documents. Do you notice anything that might be missing or incomplete?' This primes participants to spot the gaps the agent will encounter later. -->

The agent's knowledge base contains 5 support documents. Some are complete, some have deliberate gaps — just like real documentation.

In [2]:
from src.data.knowledge_base import KNOWLEDGE_BASE

for doc in KNOWLEDGE_BASE:
    print(f"\n{'='*60}")
    print(f"  {doc['doc_id']}: {doc['title']}")
    print(f"{'='*60}")
    print(doc['content'][:500] + '...' if len(doc['content']) > 500 else doc['content'])


  KB-001: Exporting Reports: Mileage, Fuel & Compliance
Meridian Fleet Solutions — Internal Support Wiki
Document: Exporting Reports: Mileage, Fuel & Compliance
Last updated: 2026-03-15

Overview
Meridian's reporting engine allows fleet operators to generate and export mileage, fuel consumption, and regulatory compliance reports directly from the Fleet Dashboard. Reports can be exported as PDF, CSV, or Excel files depending on the customer's subscription tier.

Step-by-Step Export Instructions
1. Log in to the Meridian Fleet Dashboard at dashboard.me...

  KB-002: GPS Device Troubleshooting: Offline & Signal Loss
Meridian Fleet Solutions — Internal Support Wiki
Document: GPS Device Troubleshooting: Offline & Signal Loss
Last updated: 2026-02-28

Overview
Meridian GPS tracking devices (models MF-4000 and MF-5000 series) communicate via cellular LTE with GPS positioning. Devices ping the Meridian cloud every 60 seconds under normal operation. This document covers troubleshooting steps w

## 3. Mock Database

<!-- FACILITATOR: Mock Database — Quick check: ensure participants see how customer and device records are structured. Ask: 'Why might an agent need this data in addition to the knowledge base?' -->

Customer records and employee directory — the structured data the agent can query at runtime.

In [3]:
import json
from src.data.databases import CUSTOMER_DB, EMPLOYEE_DIRECTORY

for cid, record in CUSTOMER_DB.items():
    print(f"\n{cid}: {record['company_name']} ({record['subscription_tier']})")
    print(f"  Contact: {record['contact_name']} — {record['contact_role']}")
    print(f"  Status: {record['account_status']} | Vehicles: {record['vehicle_count']}")

print(f"\n{'='*60}")
print("  Employee Directory")
print(f"{'='*60}")
for emp in EMPLOYEE_DIRECTORY:
    print(f"  {emp['name']} — {emp['role']} ({emp['authority_scope']})")


CUST-1001: Northgate Logistics (Enterprise)
  Contact: James Whitfield — CFO
  Status: ACTIVE | Vehicles: 250

CUST-1002: Pinecrest Haulage (Pro)
  Contact: Laura Chen — Fleet Manager
  Status: ACTIVE | Vehicles: 75

CUST-1003: Summit Regional Transport (Pro)
  Contact: Derek Owusu — Operations Director
  Status: ACTIVE | Vehicles: 120

CUST-1004: Ridgeline Express (Basic)
  Contact: Monica Alvarez — IT Administrator
  Status: ACTIVE | Vehicles: 30

CUST-1005: Greenfield Carriers (Pro)
  Contact: Tom Brennan — Logistics Manager
  Status: SUSPENDED | Vehicles: 95

  Employee Directory
  Dana Okafor — Billing Manager (Disputes under $500)
  Marcus Thiele — Senior Account Manager (Enterprise clients, churn risk, disputes $500–$5,000)
  Priya Nair — VP Customer Success (CRITICAL escalations, VP approval required)


## 4. Tool Registry

<!-- FACILITATOR: Tool Registry — Highlight that every tool prints a visible trace. Ask: 'Which of these tools would a traditional chatbot have access to?' (Answer: none of them.) -->

The agent has 7 tools: `search_kb`, `query_customer_db`, `send_email`, `escalate`, `request_info`, `check_sla_status`, and `check_open_cases`. Each tool prints a trace when called.

In [4]:
import json
from src.tools.registry import TOOL_SCHEMAS

for schema in TOOL_SCHEMAS:
    print(f"\n  {schema['name']}: {schema['description']}")
    params = schema.get('parameters', {}).get('properties', {})
    for pname, pdef in params.items():
        print(f"    - {pname}: {pdef.get('description', pdef.get('type', ''))}")


  search_kb: Search the knowledge base for relevant support documentation. Returns top-k matching document chunks with similarity scores.
    - query: Natural language search query

  query_customer_db: Look up customer account information from the database. Returns account status, subscription tier, device information, billing history, and other fields.
    - customer_id: The customer ID (e.g., CUST-1001)
    - fields: Optional list of specific fields to return. If omitted, returns all fields.

  send_email: Draft and display an email response to the customer. Does not actually send.
    - to: Recipient email address
    - subject: Email subject line
    - body: Email body text

  escalate: Escalate a ticket to a specific team member when the issue exceeds the agent's authority or requires human judgment. Produces a structured escalation artifact.
    - to_person: Name of the person to escalate to (e.g., 'Marcus Thiele')
    - summary: Structured summary of the issue, steps taken, an

## 5. The Frustrated Chatbot

<!-- FACILITATOR: The Frustrated Chatbot — PAUSE HERE. Give participants a full minute to read the chatbot's response. Ask: 'Would you trust this answer if you were the customer? What is it missing?' Collect two or three responses before continuing. -->

Let's see what a plain chatbot does with our active ticket — no tools, no reasoning loop, just a single response.

In [5]:
from src.data.tickets import ACTIVE_TICKET
from src.cells.contrast import run_frustrated_chatbot, CHATBOT_DEMO_RESPONSES

print(f"Ticket: {ACTIVE_TICKET['ticket_id']} — {ACTIVE_TICKET['subject']}\n")
chatbot_response = run_frustrated_chatbot(
    ACTIVE_TICKET,
    demo_response=CHATBOT_DEMO_RESPONSES[ACTIVE_TICKET['ticket_id']]
)

Ticket: T3 — Invoice overcharged + device offline + compliance report due

╔══════════════════════════════════════════════════════════╗
║  🤖 CHATBOT RESPONSE (no tools, no reasoning loop)        ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  "Thank you for contacting Meridian Fleet Solutions      ║
║   support. We apologize for the inconvenience regarding  ║
║   your invoice. Please review your billing statement in  ║
║   the customer portal and let us know if you still see a ║
║   discrepancy. We're happy to help!"                     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝


## 6. RAG in Isolation

<!-- FACILITATOR: RAG in Isolation — Pause after the retrieval result is displayed. Ask: 'We got a document back — does that count as a solution, or just a lookup?' Reinforce the label: 'This is a lookup, not a decision.' -->

Now let's add retrieval — the system finds relevant documents, but still doesn't reason or act on them.

In [6]:
from src.data.tickets import ACTIVE_TICKET
from src.cells.contrast import run_rag_only

rag_results = run_rag_only(ACTIVE_TICKET)

[TOOL: search_kb] Query: "Invoice overcharged + device offline + compliance report due"
[TOOL: search_kb] Result 1: KB-003 "Standard Customer Billing & In..." (score: 0.62)
[TOOL: search_kb] Result 2: KB-003 "Standard Customer Billing & In..." (score: 0.41)
[TOOL: search_kb] Result 3: KB-001 "Exporting Reports: Mileage, Fu..." (score: 0.39)
╔══════════════════════════════════════════════════════════╗
║  🔍 RAG-ONLY RETRIEVAL (no reasoning, no verification)    ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Query: "Invoice overcharged + device offline + co..."   ║
║                                                          ║
║  Result 1: KB-003 (score: 0.62)                          ║
║  "Standard Customer Billing & Invoice Disputes"          ║
║                                                          ║
║  Result 2: KB-003 (score: 0.41)                          ║
║  "Standard Customer Billing & Invoice Dispute

## 7. The Agent Loop

<!-- FACILITATOR: The Agent Loop — This is the core of the exercise. Walk through the printed trace line by line. Pause after each Plan/Act/Observe/Reflect cycle and ask: 'What did the agent just decide, and why?' Highlight any confidence-score changes and re-planning moments. -->

Now the full agent — it plans, acts, observes, and reflects in a loop until the ticket is resolved.

In [7]:
from src.data.tickets import ACTIVE_TICKET
from src.agent.loop import run_agent
from src.agent.demo_responses import get_demo_responses

agent_state = run_agent(
    ACTIVE_TICKET,
    demo_responses=get_demo_responses(ACTIVE_TICKET['ticket_id'])
)

[AGENT THINKING] This ticket has three issues: a $700 billing overcharge, offline devices, and a DOT compliance report deadline. I need to tackle billing first since the account appears suspended. Let me search the knowledge base for invoice dispute procedures.
[AGENT STATUS]   Confidence: 50% | Status: INVESTIGATING
[AGENT ACTION]   Calling search_kb(query="invoice dispute billing overcharge")
[DISPATCH] Calling tool: search_kb
[DISPATCH] Arguments: {"query": "invoice dispute billing overcharge"}
[TOOL: search_kb] Query: "invoice dispute billing overcharge"
[TOOL: search_kb] Result 1: KB-003 "Standard Customer Billing & In..." (score: 0.68)
[TOOL: search_kb] Result 2: KB-003 "Standard Customer Billing & In..." (score: 0.47)
[TOOL: search_kb] Result 3: KB-005 "Enterprise SLA Terms & Contrac..." (score: 0.36)
[AGENT OBSERVE]  Result: [{"doc_id": "KB-003", "title": "Standard Customer Billing & Invoice Disputes", "chunk_index": 0, "text": "Meridian Fleet Solutions \u2014 Internal Support 

## 8. Side-by-Side Comparison

<!-- FACILITATOR: Side-by-Side Comparison — Let the visual speak for itself for 30 seconds, then ask: 'What stands out when you compare the two columns?' Draw attention to the number of steps and the quality of the final output. -->

Let's compare the chatbot's single response against the agent's multi-step trace.

In [8]:
from src.data.tickets import ACTIVE_TICKET
from src.cells.comparison import run_comparison

run_comparison(ACTIVE_TICKET)

╔══════════════════════════════════════════════════════════╗
║  🤖 CHATBOT RESPONSE (no tools, no reasoning loop)        ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  "Thank you for contacting Meridian Fleet Solutions      ║
║   support. We apologize for the inconvenience regarding  ║
║   your invoice. Please review your billing statement in  ║
║   the customer portal and let us know if you still see a ║
║   discrepancy. We're happy to help!"                     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
[AGENT THINKING] Complex ticket with 3 intents: billing, device, and contract question. Addressing each systematically.
[AGENT STATUS]   Confidence: 15% | Status: INVESTIGATING
[AGENT ACTION]   Calling search_kb(query="invoice dispute")
[DISPATCH] Calling tool: search_kb
[DISPATCH] Arguments: {"query": "invoice dispute"}
[TOOL:

## 9. Ticket Variations

<!-- FACILITATOR: Ticket Variations — Invite participants to change ACTIVE_TICKET and re-run the agent loop. Suggest T4 for the self-correction demo and T5 for the escalation demo. Allow 2–3 minutes of free exploration. -->

Try different tickets to see how the agent adapts. Change the ticket ID below and re-run sections 5–8.

| Ticket | Complexity | Teaching purpose |
|--------|-----------|------------------|
| T1 | Low | Happy path — every ticket is multi-tool |
| T2 | Medium | Partial KB match forces DB verification |
| T3 | High | Multi-intent "money shot" — 3+ tool chain |
| T4 | Mismatch | Confidence collapse — agent catches wrong retrieval |
| T5 | Critical | Authority-boundary escalation |

In [9]:
from src.data.tickets import TICKETS

# ============================================================
# CHANGE THIS LINE to try a different ticket (T1–T5)
# ============================================================
ACTIVE_TICKET = TICKETS["T4"]

print(f"Active ticket: {ACTIVE_TICKET['ticket_id']} — {ACTIVE_TICKET['subject']}")
print(f"Complexity: {ACTIVE_TICKET['complexity']}")
print(f"\nNow re-run sections 5–8 above to see how the agent handles this ticket.")

Active ticket: T4 — Need info on cancellation policy
Complexity: Mismatch

Now re-run sections 5–8 above to see how the agent handles this ticket.


## 10. "Next Morning" Re-Entry

<!-- FACILITATOR: Next-Morning Re-Entry (optional) — If time permits, run this cell to show stateful persistence. Ask: 'How is this different from starting a new chat session?' Emphasise that the agent remembers every prior step. -->

The next day, Marcus Thiele replies with new information. The agent resumes with full prior context — no starting from scratch.

In [10]:
from src.cells.reentry import resume_agent, NEXT_MORNING_MESSAGE, REENTRY_DEMO_RESPONSES

updated_state = resume_agent(
    agent_state,
    NEXT_MORNING_MESSAGE['body'],
    demo_responses=REENTRY_DEMO_RESPONSES
)

╔══════════════════════════════════════════════════════════╗
║  📬 NEW MESSAGE RECEIVED — Resuming agent...           ║
╠══════════════════════════════════════════════════════════╣
║  From: Marcus Thiele, Senior Account Manager      ║
║  "I've reviewed the Northgate Logistics account. ..." ║
╠══════════════════════════════════════════════════════════╣
║  Prior context: 5 steps from previous session          ║
║  Resuming PLAN → ACT → OBSERVE → REFLECT loop...       ║
╚══════════════════════════════════════════════════════════╝
[AGENT THINKING] Marcus confirmed the billing error and issued a credit memo (#CM-2024-0847) for $2,340. I need to draft a professional confirmation email to the customer referencing the credit memo and the upcoming Thursday call with their CFO.
[AGENT STATUS]   Confidence: 95% | Status: RESOLVED
[AGENT ACTION]   Calling send_email(to="j.whitfield@northgatelogistics.com", subject="Re: Q1 Billing Dispute — Credit Memo Issued", body="Dear Mr. Whitfield,

Thank you f

## 11. Reflection

<!-- FACILITATOR: Reflection Prompts — Switch from demo to discussion. Read each question aloud and give the group 1–2 minutes per question. Close by asking participants to name one concrete next step they will take back to their team. -->

## Reflection & Discussion

Take a few minutes to discuss the following questions with your group. They move from what you just observed to what it could mean for your organisation.

---

### 1. Observation
**What did the agent do differently from the chatbot when handling the same ticket?**
Think about the number of steps, the sources of information it consulted, and the structure of its final response.

### 2. Mechanism
**Why did the agent need to call multiple tools, and could a chatbot have done the same?**
Consider what it means for a system to plan its own sequence of actions rather than producing a single response.

### 3. Self-Correction
**What happened when the agent retrieved the wrong document?** **How did it detect the mistake and recover?**
Pay attention to the confidence score and the moment the agent chose to re-query instead of trusting its first result.

### 4. Authority & Escalation
**How did the agent decide to escalate rather than attempt a resolution on its own?**
Think about the role of policy rules, authorisation limits, and the structured escalation artifact it produced.

### 5. Business Impact
**Where in your own organisation could an agentic system like this create the most value?**
Consider processes that involve multiple data sources, decision steps, or handoffs between people.

### 6. Risk & Governance
**What risks do you see in deploying an agent that makes autonomous decisions in customer-facing scenarios?**
Think about accountability, error handling, data privacy, and the boundary between automation and human judgement.

### 7. Next Steps
**What would you need to evaluate before piloting agentic AI in your team?**
Consider data readiness, process documentation, success metrics, and the change-management challenges you would face.